In [1]:
import sys
import os

from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as ss

In [2]:
from kmer_model import KmerLinearRegressor

In [3]:
src_file = "../../predictor/regression_multiple/UTR5_zscores_replicateagg.csv"
src_file_code = os.path.basename(src_file).partition("_")[0]
data = pd.read_csv(src_file)
data

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
0,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c1,train,52.125480,52.593756,49.272745,47.834227,2.459879,2.368279,0.091601,0.779921,0.117449
1,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c17,train,46.974545,51.103420,46.791818,34.483872,2.383516,2.368279,0.015238,0.129738,0.117449
2,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c2,train,50.231043,56.194372,60.956634,48.531620,2.499222,2.368279,0.130943,1.114897,0.117449
3,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c4,train,72.587265,56.853170,47.180959,36.766505,2.225536,2.368279,-0.142742,-1.215358,0.117449
4,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c6,train,63.071485,42.937131,41.591609,35.795388,2.273239,2.368279,-0.095039,-0.809197,0.117449
...,...,...,...,...,...,...,...,...,...,...,...,...
108040,TTTTTTCCCTTCTCCTGGCAATCCCTTCCGCTTCCCCGGCTCCCGG...,c1,train,24.176383,24.721123,32.529211,32.751569,2.646848,2.598530,0.048317,1.156046,0.041795
108041,TTTTTTCCCTTCTCCTGGCAATCCCTTCCGCTTCCCCGGCTCCCGG...,c17,train,43.297184,67.070795,58.874796,63.473836,2.612441,2.598530,0.013911,0.332836,0.041795
108042,TTTTTTCCCTTCTCCTGGCAATCCCTTCCGCTTCCCCGGCTCCCGG...,c2,train,27.537770,22.539731,29.272644,30.646646,2.573000,2.598530,-0.025530,-0.610841,0.041795
108043,TTTTTTCCCTTCTCCTGGCAATCCCTTCCGCTTCCCCGGCTCCCGG...,c4,train,31.382535,28.750843,30.414356,34.204236,2.540595,2.598530,-0.057935,-1.386167,0.041795


In [4]:
data_unique = data[["seq", "cell_type", "fold", "mass_center"]].drop_duplicates()
train = data_unique[data_unique["fold"] == "train"]
test = data_unique[data_unique["fold"] == "test"]

In [5]:
train_averaged = train.groupby("seq")["mass_center"].mean().reset_index()
test_averaged = test.groupby("seq")["mass_center"].mean().reset_index()

## Defining cell types

In [6]:
cell_types = sorted(data["cell_type"].unique())
cell_types

['c1', 'c17', 'c2', 'c4', 'c6']

## Creating regression models

In [7]:
pred_results = {}
metrics_results = {}

for kmer_length in (1, 2, 3):
    metrics_results[kmer_length] = metrics = defaultdict(dict)
    regressor = KmerLinearRegressor(complement=False, kmer_length=kmer_length,
                                    linreg_kws=dict(random_state=0))
    regressor.fit(train_averaged["seq"], train_averaged["mass_center"])
    test_averaged["pred_mass_center"] = regressor.predict(test_averaged["seq"])
    metrics["r"]["all_avg"] = ss.pearsonr(test_averaged["mass_center"], test_averaged["pred_mass_center"]).statistic
    metrics["rho"]["all_avg"] = ss.spearmanr(test_averaged["mass_center"], test_averaged["pred_mass_center"]).statistic

    test_copy = test.copy()
    test_copy["pred_mass_center"] = -1.0
    for ct, ctdf in train.groupby(by="cell_type"):
        regressor = KmerLinearRegressor(complement=False, kmer_length=kmer_length,
                                        linreg_kws=dict(random_state=0))
        regressor.fit(ctdf["seq"], ctdf["mass_center"])
        pred = regressor.predict(test_copy[test_copy["cell_type"] == ct]["seq"])
        test_copy.loc[test_copy["cell_type"] == ct, "pred_mass_center"] = pred

    pred_results[("pred_mass_center", f"k={kmer_length}")] = test_copy["pred_mass_center"]

    g = test_copy.groupby("seq")
    metrics["r"]["all"] = ss.pearsonr(g["mass_center"].mean(), g["pred_mass_center"].mean()).statistic
    metrics["rho"]["all"] = ss.spearmanr(g["mass_center"].mean(), g["pred_mass_center"].mean()).statistic
    for ct, ctdf in test_copy.groupby(by="cell_type"):
        metrics["r"][ct] = ss.pearsonr(ctdf["mass_center"], ctdf["pred_mass_center"]).statistic
        metrics["rho"][ct] = ss.spearmanr(ctdf["mass_center"], ctdf["pred_mass_center"]).statistic

In [8]:
pred_results_df = pd.DataFrame(pred_results).sort_index(axis=1)
pred_results_df.insert(0, "seq", test["seq"])
pred_results_df.insert(1, "cell_type", test["cell_type"])
pred_results_df.insert(2, "mass_center", test["mass_center"])
pred_results_df

seq cell_type  \
                                                                      
205     AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...        c1   
206     AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...       c17   
207     AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...        c2   
208     AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...        c4   
209     AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...        c6   
...                                                   ...       ...   
107975  TTTTCCCAGGGCGTGGGCTTGCCCCGCGCGTGTCTGTGGAGGGCGG...        c1   
107976  TTTTCCCAGGGCGTGGGCTTGCCCCGCGCGTGTCTGTGGAGGGCGG...       c17   
107977  TTTTCCCAGGGCGTGGGCTTGCCCCGCGCGTGTCTGTGGAGGGCGG...        c2   
107978  TTTTCCCAGGGCGTGGGCTTGCCCCGCGCGTGTCTGTGGAGGGCGG...        c4   
107979  TTTTCCCAGGGCGTGGGCTTGCCCCGCGCGTGTCTGTGGAGGGCGG...        c6   

       mass_center pred_mass_center                      
                                k=1       k=2       k=3  
205       2.465254         2.498934  2.603037  2.540197  
206       2.322471         2.480798  2.623532  2.573754  
207       2.464814         2.520651  2.583812  2.547669  
208       2.532891         2.480016  2.567928  2.514297  
209       2.323651         2.485782  2.564377  2.492031  
...            ...              ...       ...       ...  
107975    2.503601         2.521484  2.551341  2.545752  
107976    2.479266         2.513857  2.565160  2.599175  
107977    2.483830         2.507846  2.527614  2.521249  
107978    2.419081         2.508624  2.537619  2.554979  
107979    2.360696         2.507585  2.506918  2.506774  

[10800 rows x 6 columns]

In [9]:
pred_results_df.to_parquet("models_kmer_preds_utr5.parquet")

In [10]:
import json

with open("models_kmer_utr5.json", "wt") as outfile:
    json.dump(metrics_results, outfile)